In [1]:
import os
import json
import torch
import numpy as np
import faiss
from tqdm import tqdm

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments
)

from sentence_transformers import SentenceTransformer, CrossEncoder
import evaluate

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_MODEL = "dslim/bert-base-NER"
RETRIEVER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

MAX_LENGTH = 128
TOP_K = 5

METRIC = evaluate.load("seqeval")

In [3]:
dataset = load_dataset("conll2003")

train_data = dataset["train"].select(range(2000))
val_data = dataset["validation"].select(range(500))

label_names = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_names)

Using the latest cached version of the dataset since conll2003 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'conll2003' at C:\Users\LENOVO\.cache\huggingface\datasets\conll2003\conll2003\1.0.0\9a4d16a94f8674ba3466315300359b0acd891b68b6c8743ddf60b9c702adce98 (last modified on Fri Mar  6 12:40:28 2026).


In [4]:
class DenseRetriever:
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name, device=str(DEVICE))
        self.index = None
        self.texts = []

    def build(self, texts):
        emb = self.model.encode(texts, convert_to_numpy=True)
        faiss.normalize_L2(emb)
        dim = emb.shape[1]

        self.index = faiss.IndexFlatIP(dim)
        self.index.add(emb)

        self.texts = texts

    def query(self, text, k=5):
        q = self.model.encode([text], convert_to_numpy=True)
        faiss.normalize_L2(q)

        D, I = self.index.search(q, k)
        return [self.texts[i] for i in I[0]]

In [5]:
import pickle

KB_PATH = "kb.pkl"
INDEX_PATH = "faiss.index"
META_PATH = "faiss_meta.pkl"

# LOAD OR BUILD KB
if os.path.exists(KB_PATH):
    print("Loading cached KB...")
    with open(KB_PATH, "rb") as f:
        KB = pickle.load(f)
else:
    print("Building KB...")
    wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:50000]")
    KB = [x["text"][:300] for x in wiki if len(x["text"]) > 100]

    with open(KB_PATH, "wb") as f:
        pickle.dump(KB, f)

# INIT RETRIEVER
retriever = DenseRetriever(RETRIEVER_MODEL)

# LOAD OR BUILD FAISS
if os.path.exists(INDEX_PATH) and os.path.exists(META_PATH):
    print("Loading FAISS index...")
    retriever.index = faiss.read_index(INDEX_PATH)

    with open(META_PATH, "rb") as f:
        retriever.texts = pickle.load(f)
else:
    print("Building FAISS index...")
    retriever.build(KB)

    faiss.write_index(retriever.index, INDEX_PATH)
    with open(META_PATH, "wb") as f:
        pickle.dump(KB, f)

# CROSS ENCODER (always fast)
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)

Building KB...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

20231101.en/train-00016-of-00041.parquet:   0%|          | 0.00/503M [00:00<?, ?B/s]

c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\datasets--wikimedia--wikipedia. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


20231101.en/train-00017-of-00041.parquet:   0%|          | 0.00/231M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building FAISS index...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
class RANER:
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
        self.model = AutoModelForTokenClassification.from_pretrained(
            BASE_MODEL,
            num_labels=num_labels
        ).to(DEVICE)

    def retrieve(self, span, context):
        query = f"{span} in {context}"
        docs = retriever.query(query, k=TOP_K)

        pairs = [[query, d] for d in docs]
        scores = cross_encoder.predict(pairs)

        best = docs[int(np.argmax(scores))]
        return best

    def build_input(self, tokens):
        sentence = " ".join(tokens)

        enriched = sentence
        for tok in tokens:
            if len(tok) < 3:
                continue

            doc = self.retrieve(tok, sentence)
            enriched += f" [KNOW] {doc}"

        return enriched

    def tokenize_and_align(self, tokens, labels):
        text = self.build_input(tokens)

        enc = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH
        )

        word_ids = enc.word_ids()

        aligned = []
        prev = None
        for w in word_ids:
            if w is None:
                aligned.append(-100)
            elif w != prev:
                aligned.append(labels[w])
            else:
                aligned.append(labels[w])
            prev = w

        enc["labels"] = aligned
        return enc

In [ ]:
from datasets import load_from_disk

TRAIN_PATH = "train_raner"
VAL_PATH = "val_raner"

raner = RANER()

if os.path.exists(TRAIN_PATH) and os.path.exists(VAL_PATH):
    print("Loading cached datasets...")
    train_dataset = load_from_disk(TRAIN_PATH)
    val_dataset = load_from_disk(VAL_PATH)

else:
    print("Processing datasets...")

    def preprocess(example):
        return raner.tokenize_and_align(example["tokens"], example["ner_tags"])

    train_dataset = train_data.map(
        preprocess,
        batched=False
    )

    val_dataset = val_data.map(
        preprocess,
        batched=False
    )

    train_dataset.save_to_disk(TRAIN_PATH)
    val_dataset.save_to_disk(VAL_PATH)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing datasets...


Parameter 'function'=<function preprocess at 0x000001F18B5A4680> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./raner",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

trainer = Trainer(
    model=raner.model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

In [ ]:
baseline_model = AutoModelForTokenClassification.from_pretrained(BASE_MODEL).to(DEVICE)
baseline_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

In [ ]:
def word_level(tokens, preds):
    words = []
    labels = []

    cur = ""
    cur_lab = None

    for tok, p in zip(tokens, preds):
        lab = label_names[p]

        if tok in ["[CLS]", "[SEP]"]:
            continue

        if tok.startswith("##"):
            cur += tok[2:]
        else:
            if cur:
                words.append(cur)
                labels.append(cur_lab)
            cur = tok
            cur_lab = lab

    if cur:
        words.append(cur)
        labels.append(cur_lab)

    return labels


def evaluate_model(model, tokenizer, dataset):
    all_preds = []
    all_labels = []

    for item in dataset:
        tokens = item["tokens"]
        sentence = " ".join(tokens)

        inputs = tokenizer(sentence, return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            out = model(**inputs)

        preds = torch.argmax(out.logits, dim=-1)[0].cpu().numpy()

        toks = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

        pred_labels = word_level(toks, preds)

        true_labels = [label_names[x] for x in item["ner_tags"]]

        m = min(len(pred_labels), len(true_labels))

        all_preds.append(pred_labels[:m])
        all_labels.append(true_labels[:m])

    return METRIC.compute(predictions=all_preds, references=all_labels)

In [ ]:
baseline_results = evaluate_model(baseline_model, baseline_tokenizer, val_data)
raner_results = evaluate_model(raner.model, raner.tokenizer, val_data)

print("\nBASELINE:", baseline_results)
print("\nRA-NER:", raner_results)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix
import seaborn as sns

# GET TRAINING LOGS
logs = trainer.state.log_history
df_logs = pd.DataFrame(logs)

df_logs.to_csv("training_logs.csv", index=False)

# EXTRACT METRICS
train_loss = df_logs[df_logs["loss"].notna()]
eval_metrics = df_logs[df_logs["eval_loss"].notna()]

# PLOT LOSS
plt.figure()
plt.plot(train_loss["step"], train_loss["loss"])
plt.title("Training Loss")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.savefig("train_loss.png")
plt.show()

plt.figure()
plt.plot(eval_metrics["step"], eval_metrics["eval_loss"])
plt.title("Validation Loss")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.savefig("val_loss.png")
plt.show()

# FINAL METRICS
baseline_results = evaluate_model(baseline_model, baseline_tokenizer, val_data)
raner_results = evaluate_model(raner.model, raner.tokenizer, val_data)

metrics_df = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1", "Accuracy"],
    "Baseline": [
        baseline_results["overall_precision"],
        baseline_results["overall_recall"],
        baseline_results["overall_f1"],
        baseline_results["overall_accuracy"]
    ],
    "RA-NER": [
        raner_results["overall_precision"],
        raner_results["overall_recall"],
        raner_results["overall_f1"],
        raner_results["overall_accuracy"]
    ]
})

metrics_df["Improvement"] = metrics_df["RA-NER"] - metrics_df["Baseline"]

metrics_df.to_csv("final_metrics.csv", index=False)
print(metrics_df)

# BAR PLOT
metrics_df.set_index("Metric")[["Baseline", "RA-NER"]].plot(kind="bar")
plt.title("Baseline vs RA-NER")
plt.savefig("comparison_bar.png")
plt.show()

# CONFUSION MATRIX
all_preds = []
all_labels = []

for item in val_data:
    tokens = item["tokens"]
    sentence = " ".join(tokens)

    inputs = raner.tokenizer(sentence, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        out = raner.model(**inputs)

    preds = torch.argmax(out.logits, dim=-1)[0].cpu().numpy()
    toks = raner.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    pred_labels = word_level(toks, preds)
    true_labels = [label_names[x] for x in item["ner_tags"]]

    m = min(len(pred_labels), len(true_labels))

    all_preds.extend(pred_labels[:m])
    all_labels.extend(true_labels[:m])

# MAP LABELS TO IDS
label_to_id = {l:i for i,l in enumerate(label_names)}
y_true = [label_to_id[x] for x in all_labels]
y_pred = [label_to_id[x] for x in all_preds]

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=False)
plt.title("Confusion Matrix")
plt.savefig("confusion_matrix.png")
plt.show()

In [ ]:
SAVE_PATH = "./raner_final_model"

os.makedirs(SAVE_PATH, exist_ok=True)

trainer.save_model(SAVE_PATH)
raner.tokenizer.save_pretrained(SAVE_PATH)

print(f"Model saved at {SAVE_PATH}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

LOAD_PATH = "./raner_final_model"

tokenizer = AutoTokenizer.from_pretrained(LOAD_PATH)
model = AutoModelForTokenClassification.from_pretrained(LOAD_PATH).to(DEVICE)

print("Model loaded successfully")

In [ ]:
BEST_PATH = trainer.state.best_model_checkpoint
FINAL_SAVE_PATH = "./raner_best_model"

from transformers import AutoModelForTokenClassification, AutoTokenizer
import shutil
import os

os.makedirs(FINAL_SAVE_PATH, exist_ok=True)

model = AutoModelForTokenClassification.from_pretrained(BEST_PATH)
tokenizer = raner.tokenizer

model.save_pretrained(FINAL_SAVE_PATH)
tokenizer.save_pretrained(FINAL_SAVE_PATH)

print(f"Best model saved from {BEST_PATH} → {FINAL_SAVE_PATH}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

PATH = "./raner_best_model"

tokenizer = AutoTokenizer.from_pretrained(PATH)
model = AutoModelForTokenClassification.from_pretrained(PATH).to(DEVICE)

print("Best model loaded")